In [1]:
from huggingface_hub import snapshot_download
from faster_whisper import WhisperModel

import csv
import unicodedata
from pathlib import Path

from IPython.display import Audio, display

/Users/yetodo/Documents/meall/kanbior/.kanbior-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ct2-transformers-converter --model runs/whisper-fon-smoke/final \
    --output_dir runs/kanbior-asr-v00 \
    --copy_files tokenizer.json  \                        
    --quantization int8

In [3]:
path = "runs/kanbior-asr-v00"
model = WhisperModel(path, device="cpu", compute_type="int8")

[2026-11-22 20:12:22.855] [ctranslate2] [thread 145956] [warning] Tip: experimental MPS backend is available and often 2x+ faster than CPU on Apple Silicon (varies by workload and hardware). Try device="mps".


In [4]:
DATA_DIR = Path("sample-data")
with open(DATA_DIR / "transcribed.csv", newline="", encoding="utf-8") as f:
    transcripts = {row["id"]: row for row in csv.DictReader(f)}

def transcribe(clip_id):
    row = transcripts[clip_id]
    audio_path = DATA_DIR / row["audio_path"]
    display(Audio(str(audio_path)))
    segments, info = model.transcribe(str(audio_path), language="fr", beam_size=1)
    predicted = " ".join(s.text.strip() for s in segments)
    print(f"Clip:      {clip_id}  ({row['speaker_id']}, {row['duration']} s)")
    print("Reference:", unicodedata.normalize("NFC", row["transcription"]))
    print("Predicted:", unicodedata.normalize("NFC", predicted))



In [ ]:
transcribe("tr_000222")

Clip:      tr_000230  (JML_S517, 6.42 s)
Reference: É ká jɛ́ xá din bɔ mǐ hɛn ɔ́ mǐ na yi "marquis" bó ná yi ɖú we ɖo axɔ́súzángbe hwelɛ́kɔ a ?
Predicted: Écadéxad mɛ bɔ mɛ hɛ́n ɔ́ mɛ ná yi mɛ́ kí bɔná yi ɖu wɛ ɖo mɛ́ nuzɔn yɛ xɔzɔn gbɛ bɛ́ lɛ́ɛ kɔ́ a?
